In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import urllib

# 1. CẤU HÌNH KẾT NỐI SQL SERVER

In [ ]:
# 1. CẤU HÌNH KẾT NỐI SQL SERVER
server = 'YOUR_SERVER_NAME' 
database = 'SEA_Economic_DB'
params = urllib.parse.quote_plus(f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};Trusted_Connection=yes;')
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}", fast_executemany=True)

# 2. XỬ LÝ BẢNG MASTER (METADATA)

In [ ]:
# 2. XỬ LÝ BẢNG MASTER (METADATA)
print("Đang đọc và xử lý Metadata...")
df_meta = pd.read_csv('1. Dataset/Metadata_Country.csv')
df_countries = df_meta[['Country Code', 'TableName', 'Region', 'IncomeGroup']].copy()
df_countries.columns = ['country_code', 'country_name', 'region', 'income_group']
df_countries = df_countries.where(pd.notnull(df_countries), None)

# 3. HÀM UNPIVOT & LÀM SẠCH CƠ BẢN

In [ ]:
# 3. HÀM UNPIVOT & LÀM SẠCH CƠ BẢN
def clean_data(file_path, val_col):
    df = pd.read_csv(file_path, skiprows=4)
    cols_drop = ['Country Name', 'Indicator Name', 'Indicator Code', 'Unnamed: 68']
    df = df.drop(columns=[c for c in cols_drop if c in df.columns])
    
    # UNPIVOT
    df_unpivoted = df.melt(id_vars=['Country Code'], var_name='year', value_name=val_col)
    
    df_unpivoted['year'] = pd.to_numeric(df_unpivoted['year'], errors='coerce')
    df_unpivoted = df_unpivoted.dropna(subset=['year'])
    df_unpivoted['year'] = df_unpivoted['year'].astype(int)
    
    df_unpivoted[val_col] = pd.to_numeric(df_unpivoted[val_col], errors='coerce')
    df_clean = df_unpivoted[(df_unpivoted['year'] >= 2000) & (df_unpivoted['year'] <= 2023)]
    df_clean = df_clean.rename(columns={'Country Code': 'country_code'})
    
    return df_clean.dropna(subset=[val_col]) # Bỏ dòng NULL giá trị

# 4. GỘP BẢNG VÀ THỰC HIỆN 4 BƯỚC DATA CLEANING CHUYÊN SÂU

In [ ]:
# 4. GỘP BẢNG VÀ THỰC HIỆN 4 BƯỚC DATA CLEANING CHUYÊN SÂU
# A. Xử lý GDP
df_gdp_usd = clean_data('1. Dataset/gdp_by_country.csv', 'gdp_usd')
df_gdp_pc = clean_data('1. Dataset/gdp_per_capita.csv', 'gdp_per_capita_usd')
df_gdp_final = pd.merge(df_gdp_usd, df_gdp_pc, on=['country_code', 'year'], how='outer')

In [ ]:
# B. Xử lý Population & Economic Indicators
df_pop = clean_data('1. Dataset/population.csv', 'population')

df_eco = pd.merge(clean_data('1. Dataset/inflation.csv', 'inflation_rate'), 
                  clean_data('1. Dataset/unemployment.csv', 'unemployment_rate'), on=['country_code', 'year'], how='outer')
df_eco = pd.merge(df_eco, clean_data('1. Dataset/fdi_inflows.csv', 'fdi_usd'), on=['country_code', 'year'], how='outer')

## Clean data

In [ ]:
# BƯỚC 1: Kiểm tra Missing Values
print("\n1. Báo cáo Missing Values bảng GDP:")
print(df_gdp_final.isnull().sum())

In [ ]:
# BƯỚC 2: Kiểm tra và Xóa Duplicate
duplicates_count = df_gdp_final.duplicated(subset=['country_code', 'year']).sum()
df_gdp_final = df_gdp_final.drop_duplicates(subset=['country_code', 'year'], keep='first')

In [ ]:
# BƯỚC 3: Kiểm tra Outlier (Biến động GDP > 50%)
df_gdp_final = df_gdp_final.sort_values(by=['country_code', 'year'])
df_gdp_final['gdp_growth'] = df_gdp_final.groupby('country_code')['gdp_usd'].pct_change()
outliers = df_gdp_final[abs(df_gdp_final['gdp_growth']) > 0.5]
print(f"\n3. Phát hiện {len(outliers)} dòng có GDP biến động bất thường (>50%).")

# Xóa cột gdp_growth vì Schema SQL không có cột này
df_gdp_final = df_gdp_final.drop(columns=['gdp_growth'])

In [ ]:
# BƯỚC 4: Chuẩn hóa đơn vị (Đổi sang Tỷ USD)
df_gdp_final['gdp_usd'] = (df_gdp_final['gdp_usd'] / 1e9).round(2)

# Đổi FDI sang Tỷ USD cho đồng bộ
df_eco['fdi_usd'] = (df_eco['fdi_usd'] / 1e9).round(2)

# 5. ĐẨY DỮ LIỆU VÀO SQL SERVER

In [ ]:
# 5. ĐẨY DỮ LIỆU VÀO SQL SERVER
try:
    print("\nĐang bơm dữ liệu vào Database...")
    
    df_countries.to_sql('countries', con=engine, if_exists='append', index=False)
    print("✅ Đã load xong bảng 'countries'")
    
    df_gdp_final.to_sql('gdp_data', con=engine, if_exists='append', index=False)
    print("✅ Đã load xong bảng 'gdp_data'")
    
    df_pop.to_sql('population_data', con=engine, if_exists='append', index=False)
    print("✅ Đã load xong bảng 'population_data'")
    
    df_eco.to_sql('economic_indicators', con=engine, if_exists='append', index=False)
    print("✅ Đã load xong bảng 'economic_indicators'")
    
    print("\n🎉 QUÁ TRÌNH ETL HOÀN TẤT XUẤT SẮC!")
    
except Exception as e:
    print(f"\n❌ Có lỗi xảy ra: {e}")